Install lib

In [1]:
!pip install \
transformers==4.53.3 \
peft==0.17.1 \
trl==0.19.1 \
accelerate==1.8.1 \
bitsandbytes==0.46.1 \
datasets==3.6.0 \
huggingface_hub==0.34.4 \
sentencepiece

In [2]:
from huggingface_hub import login

login("HF_TOKEN")

Set up the model

In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    AutoModelForCausalLM,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, PeftModel, PeftConfig
from trl import SFTTrainer

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
output_dir = "./TinyLlama"


In [ ]:
dataset = load_dataset("Abirate/english_quotes")
dataset

DatasetDict({
    train: Dataset({
        features: ['quote', 'author', 'tags'],
        num_rows: 2508
    })
})

Format example

In [ ]:
def format_example(example):
  quote = example["quote"].strip()
  tags = ", ".join(example["tags"]) if isinstance(example["tags"], list) else str(example["tags"])
  text = f"""<bos><start_of_turn>user
    Generate relevant tags for the following quote.

    Quote:
    {quote}<end_of_turn>
    <start_of_turn>model
    {tags}<end_of_turn>"""
  return {"text": text}

In [ ]:
train_dataset = dataset["train"].map(format_example)

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
train_dataset

Dataset({
    features: ['quote', 'author', 'tags', 'text'],
    num_rows: 2508
})

This format teaches the model how to respond to a user instruction by mapping input text to the expected output.

In [ ]:
print(train_dataset[1]['text'])

<bos><start_of_turn>user
    Generate relevant tags for the following quote.

    Quote:
    “I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”<end_of_turn>
    <start_of_turn>model
    best, life, love, mistakes, out-of-control, truth, worst<end_of_turn>


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.bfloat16
)

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map = "auto",
    dtype = torch.float16
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Setting up the LoRA adapter

In [ ]:
lora_config = LoraConfig(
    r = 16,
    lora_alpha = 32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [ ]:
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    bf16=True,
    fp16=False,
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
)

Adding EOS to train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.336216
20,1.254048
30,1.052712
40,1.082369
50,1.130457
60,0.940008
70,0.930252
80,1.049688
90,1.143656
100,0.954188


TrainOutput(global_step=627, training_loss=1.0340201942163982, metrics={'train_runtime': 1128.5524, 'train_samples_per_second': 2.222, 'train_steps_per_second': 0.556, 'total_flos': 1717066574106624.0, 'train_loss': 1.0340201942163982, 'entropy': 1.1069804451295309, 'num_tokens': 275431.0, 'mean_token_accuracy': 0.7586483976670674, 'epoch': 1.0})

Save LoRA adapter

In [ ]:
model.save_pretrained(f"{model_name}/adapter")
tokenizer.save_pretrained(f"{model_name}/adapter")

('TinyLlama/TinyLlama-1.1B-Chat-v1.0/adapter/tokenizer_config.json',
 'TinyLlama/TinyLlama-1.1B-Chat-v1.0/adapter/chat_template.jinja',
 'TinyLlama/TinyLlama-1.1B-Chat-v1.0/adapter/tokenizer.json')

Share adapter on the Hub

In [ ]:
model.push_to_hub(
    "okarinn06/TinyLlama-1.1B-Chat-v1.0-LoRA-tagger",
    commit_message="Basic fine tuning",
)

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 9.03MB / 9.03MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/okarinn06/TinyLlama-1.1B-Chat-v1.0-LoRA-tagger/commit/f9fa81f82b2ec80f2b432e900cbc30c7b1535a6a', commit_message='Basic fine tuning', commit_description='', oid='f9fa81f82b2ec80f2b432e900cbc30c7b1535a6a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/okarinn06/TinyLlama-1.1B-Chat-v1.0-LoRA-tagger', endpoint='https://huggingface.co', repo_type='model', repo_id='okarinn06/TinyLlama-1.1B-Chat-v1.0-LoRA-tagger'), pr_revision=None, pr_num=None)

Load adapter from the hub

In [11]:
import torch
import json
import os
from huggingface_hub import hf_hub_download, snapshot_download
from transformers import LlamaConfig, LlamaForCausalLM, AutoTokenizer, BitsAndBytesConfig, AutoConfig, AutoModelForCausalLM
from peft import PeftModel

peft_model_id = "okarinn06/TinyLlama-1.1B-Chat-v1.0-LoRA-tagger"
base_model = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Download model
local_model_path = snapshot_download(repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0")

# Patch config.json before load
config_file = hf_hub_download(repo_id=base_model, filename="config.json")
with open(config_file) as f:
    model_cfg = json.load(f)

model_cfg["model_type"] = "llama"

patched_config_dir = "/tmp/tinyllama_patched"
os.makedirs(patched_config_dir, exist_ok=True)
with open(f"{patched_config_dir}/config.json", "w") as f:
    json.dump(model_cfg, f)

# Load config from patched folder
llama_config = LlamaConfig.from_pretrained(patched_config_dir)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant = True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = LlamaForCausalLM.from_pretrained(
    local_model_path,
    config=llama_config,
    quantization_config=bnb_config,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(local_model_path)

# Load LoRA
model = PeftModel.from_pretrained(model, peft_model_id)

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

Downloaded to: /root/.cache/huggingface/hub/models--TinyLlama--TinyLlama-1.1B-Chat-v1.0/snapshots/fe8a4ea1ffedaf415f4da2f062534de366a451e6
model_type trong config: llama


adapter_model.safetensors:   0%|          | 0.00/9.03M [00:00<?, ?B/s]

Load thành công!


Inference

In [13]:
model.eval()

def generate_tags(quote: str, max_new_tokens: int = 50) -> str:
    prompt = f"""<bos><start_of_turn>user
Generate relevant tags for the following quote.

Quote:
{quote}<end_of_turn>
<start_of_turn>model
"""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(generated, skip_special_tokens=True)
    return result.strip()

# Test quotes
test_quotes = [
    "The only way to do great work is to love what you do.",
    "In the middle of every difficulty lies opportunity.",
    "Life is what happens when you're busy making other plans.",
]

for quote in test_quotes:
    tags = generate_tags(quote)
    print(f"Quote : {quote}")
    print(f"Tags  : {tags}")
    print("-" * 60)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:447: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Quote : The only way to do great work is to love what you do.
Tags  : love, work<end_of_turn>
------------------------------------------------------------
Quote : In the middle of every difficulty lies opportunity.
Tags  : chance, opportunities<end_of_turn>
------------------------------------------------------------
Quote : Life is what happens when you're busy making other plans.
Tags  : life, planning, time<end_of_turn>
------------------------------------------------------------
